In [57]:
# ============================================================
# 🦾 MASTER DASHBOARD: All-In-One (2D Report + 3D Replay)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Ensure 3D plotting is registered
from mpl_toolkits.mplot3d import Axes3D 
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os
import re

# 1. Setup
BASE_LOG_DIR = os.path.join("..", "..", "logs")
L1 = 15.0 # cm
L2 = 15.0 # cm
MAX_ALLOWED_ERROR = 0.5 # cm

# 2. Define the Main Analysis Logic
def load_and_analyze(run_folder_path):
    clear_output(wait=True)
    
    run_name = os.path.basename(run_folder_path)
    
    # --- LOAD DATA ---
    log_files = glob.glob(os.path.join(run_folder_path, "arm_2link_log.csv"))
    if not log_files:
        print(f"❌ No data found in {run_name}")
        return
    df = pd.read_csv(log_files[0])
    df['time_sec'] = df['timestamp_ms'] / 1000.0
    
    # --- PRE-CALCULATE GEOMETRY ---
    rad_m1 = np.radians(df['motor1_pos'])
    df['elbow_x'] = L1 * np.cos(rad_m1)
    df['elbow_y'] = L1 * np.sin(rad_m1)
    
    # --- METRICS & FIX FOR INF SPEED ---
    duration = df['time_sec'].max()
    max_error = df['error'].max()
    avg_error = df['error'].mean()
    verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"
    
    total_j1_travel = np.sum(np.abs(np.diff(df['motor1_pos'])))
    total_j2_travel = np.sum(np.abs(np.diff(df['motor2_pos'])))
    
    # FIX: Calculate dt safely to avoid division by zero
    dt = df['time_sec'].diff()
    dt[dt <= 0] = 0.001  # Force min 1ms if duplicate timestamps exist
    dt = dt.fillna(0.001)

    # Skip startup noise
    max_j1_vel = (np.abs(df['motor1_pos'].diff()) / dt).iloc[5:].max()
    max_j2_vel = (np.abs(df['motor2_pos'].diff()) / dt).iloc[5:].max()

    # ========================================================
    # 📊 PART 1: STATIC ANALYSIS REPORT (The 3 Plots)
    # ========================================================
    # Create figure specifically for this run
    fig_static = plt.figure(figsize=(12, 10))
    plt.subplots_adjust(hspace=0.4)
    
    # Plot 1: Cartesian Path
    ax1 = fig_static.add_subplot(3, 1, 1)
    ax1.set_title(f"1. Cartesian Path ({run_name})", fontweight='bold')
    ax1.plot(df['target_x'], df['target_y'], 'g--', label='Target')
    ax1.plot(df['real_x'], df['real_y'], 'b-', label='Actual Tip')
    
    last = df.iloc[-1]
    ax1.plot([0, last['elbow_x'], last['real_x']], 
             [0, last['elbow_y'], last['real_y']], 'r-o', lw=2, label='Final Pose')
    ax1.scatter([0], [0], c='k', s=100, marker='s')
    ax1.axis('equal')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    ax1.set_ylabel("Y (cm)")

    # Plot 2: Error Timeline
    ax2 = fig_static.add_subplot(3, 1, 2)
    ax2.set_title("2. Tracking Error (Full Run)", fontweight='bold')
    ax2.plot(df['time_sec'], df['error'], 'r-', label='Error')
    ax2.axhline(MAX_ALLOWED_ERROR, color='k', linestyle=':', label='Limit')
    ax2.fill_between(df['time_sec'], df['error'], color='red', alpha=0.1)
    ax2.set_ylabel("Error (cm)")
    ax2.grid(True, alpha=0.3)

    # Plot 3: Landing Zoom
    ax3 = fig_static.add_subplot(3, 1, 3)
    start_zoom = duration * 0.8
    zoom_df = df[df['time_sec'] > start_zoom]
    ax3.set_title("3. Landing Zoom (Final 20%)", fontweight='bold')
    if not zoom_df.empty:
        ax3.plot(zoom_df['time_sec'], zoom_df['error'], 'r-o', markersize=4)
        ax3.axhline(0, color='k', alpha=0.5)
        y_limit = max(zoom_df['error'].max(), 0.05) * 1.2
        ax3.set_ylim(-0.01, y_limit)
    ax3.set_ylabel("Error (cm)")
    ax3.set_xlabel("Time (s)")
    ax3.grid(True, which='both', alpha=0.3)

    plt.show()

    # --- SUMMARY TABLE ---
    summary_data = {
        "Metric": ["Duration", "Max Error", "Avg Error", "Verdict", "---",
                   "J1 Total Travel", "J2 Total Travel", "J1 Max Speed", "J2 Max Speed"],
        "Value": [f"{duration:.2f} s", f"{max_error:.4f} cm", f"{avg_error:.4f} cm", verdict, "---",
                  f"{total_j1_travel:.1f}°", f"{total_j2_travel:.1f}°",
                  f"{max_j1_vel:.1f} deg/s", f"{max_j2_vel:.1f} deg/s"]
    }
    print("\n" + "="*60)
    print(f"📊 REPORT: {run_name}")
    print("="*60)
    print(pd.DataFrame(summary_data).to_string(index=False))
    print("="*60 + "\n")

    # ========================================================
    # 🎮 PART 2: 3D INTERACTIVE REPLAY
    # ========================================================
    print("👇 3D INTERACTIVE REPLAY (Use Sliders to Rotate & Move) 👇")
    
    def plot_3d_frame(frame_index, elev, azim):
        fig_3d = plt.figure(figsize=(10, 8))
        ax = fig_3d.add_subplot(111, projection='3d')
        
        row = df.iloc[frame_index]
        history_slice = df.iloc[:frame_index+1:5]
        
        ax.set_xlim(-10, 35)
        ax.set_ylim(-15, 25)
        ax.set_zlim(0, 20)
        ax.set_title(f"T={row['time_sec']:.2f}s | J1: {row['motor1_pos']:.0f}° J2: {row['motor2_pos']:.0f}°")
        
        # Draw Objects
        ax.plot([-10, 35, 35, -10, -10], [-15, -15, 25, 25, -15], [0,0,0,0,0], 'k-', alpha=0.1) # Floor
        ax.plot(df['target_x'], df['target_y'], np.zeros(len(df)), 'g--', alpha=0.3) # Target Path
        ax.plot(history_slice['real_x'], history_slice['real_y'], np.zeros(len(history_slice)), 'b-', alpha=0.6) # Trace
        
        # Arm
        arm_z = 5.0
        ax.plot([0,0], [0,0], [0, arm_z], 'k-', lw=6, alpha=0.5)
        ax.plot([0, row['elbow_x']], [0, row['elbow_y']], [arm_z, arm_z], color='gray', lw=4, solid_capstyle='round')
        ax.plot([row['elbow_x'], row['real_x']], [row['elbow_y'], row['real_y']], [arm_z, arm_z], color='red', lw=4, solid_capstyle='round')
        ax.plot([0, row['elbow_x'], row['real_x']], [0, row['elbow_y'], row['real_y']], [0, 0, 0], 'k-', lw=2, alpha=0.2) # Shadow
        
        ax.view_init(elev=elev, azim=azim)
        plt.show()

    # Sliders
    slider_time = widgets.IntSlider(min=0, max=len(df)-1, step=10, value=0, description='Time:', layout=widgets.Layout(width='60%'))
    slider_elev = widgets.IntSlider(min=0, max=90, step=5, value=30, description='Elevation:')
    slider_azim = widgets.IntSlider(min=0, max=360, step=5, value=270, description='Rotation:')
    
    # UI Layout
    ui = widgets.VBox([slider_time, widgets.HBox([slider_elev, slider_azim])])
    out = widgets.interactive_output(plot_3d_frame, {'frame_index': slider_time, 'elev': slider_elev, 'azim': slider_azim})
    
    display(ui, out)

# 3. Setup the Dropdown & Main Output Container
all_folders = [f.path for f in os.scandir(BASE_LOG_DIR) if f.is_dir()]
arm_folders = sorted(
    [f for f in all_folders if "arm_run" in os.path.basename(f)],
    key=lambda x: int(re.search(r"arm_run_?(\d+)", os.path.basename(x)).group(1)) 
                  if re.search(r"arm_run_?(\d+)", os.path.basename(x)) else 0
)

# Output widget to hold the dashboard
dashboard_output = widgets.Output()

if not arm_folders:
    print("❌ No 'arm_run' folders found.")
else:
    # Create Dropdown
    dropdown = widgets.Dropdown(
        options=[(os.path.basename(f), f) for f in arm_folders],
        value=arm_folders[-1], # Default to latest
        description='Select Run:',
        layout=widgets.Layout(width='50%')
    )
    
    # Define Event Handler
    def on_run_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with dashboard_output:
                load_and_analyze(change['new'])

    dropdown.observe(on_run_change)
    
    # Display Widgets
    display(dropdown)
    display(dashboard_output)
    
    # Trigger First Load
    with dashboard_output:
        load_and_analyze(arm_folders[-1])

Dropdown(description='Select Run:', index=5, layout=Layout(width='50%'), options=(('arm_run_1', '..\\..\\logs\…

Output()